# CAPE-TTA V7 真实 LoRA 更新与回滚开发验证

选择 T4 GPU → 保存 → 全部运行。默认 task 0 / init 0，每个分支最多 220 步，共三个分支：冻结、更新、恢复回放。

固定源代码 5ee6c763c94ffc2c575bee6314a90187676fbf07。一个 optimizer step，rank 4，学习率 1e-5。检查非零参数变化、冻结权重哈希、恢复参数和轨迹。训练目标仍来自 V5 Oracle；没有独立校准，不是正式性能实验。

GPU 训练兼容性与显存尚未验证；失败也会下载诊断 ZIP。不要重复运行旧 V6B。结束后上传 ZIP。


In [ ]:
# V7 actual LoRA update development check: one cell, automatic environment setup and ZIP download.
import datetime as dt
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

SOURCE_SHA = "5ee6c763c94ffc2c575bee6314a90187676fbf07"
SOURCE = Path("/content/cape-source") / SOURCE_SHA
ROOT = Path("/content/cape-full-horizon") / ("lora-update-" + dt.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8])
ROOT.mkdir(parents=True, exist_ok=False)
PY = Path("/content/cape-env-v3/bin/python")
LIBERO = Path("/content/LIBERO-8f1084e3132a")
state = {"status": "running", "scientific_result": False, "source_sha": SOURCE_SHA}
log = (ROOT / "launcher.log").open("w", encoding="utf-8")

def run(command, env=None):
    line = "$ " + " ".join(map(str, command))
    print(line, flush=True)
    log.write(line + "\n"); log.flush()
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        status = proc.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)

try:
    smi = shutil.which("nvidia-smi")
    if smi is None:
        raise RuntimeError("请选择：代码执行程序 → 更改运行时类型 → T4 GPU → 保存，再运行。")
    run([smi, "--query-gpu=name,memory.total", "--format=csv,noheader"])
    if not SOURCE.exists():
        SOURCE.mkdir(parents=True)
        run(["git", "-C", SOURCE, "init"])
        run(["git", "-C", SOURCE, "remote", "add", "origin", "https://github.com/weichezhang/CAPE-TTA.git"])
        run(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", SOURCE_SHA])
        run(["git", "-C", SOURCE, "checkout", "--detach", SOURCE_SHA])
    actual = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    if actual != SOURCE_SHA:
        raise RuntimeError("Unexpected source revision: " + actual)
    subprocess.run(["git", "-C", str(SOURCE), "diff", "--quiet", "HEAD"], check=True)
    warm = False
    if PY.is_file() and LIBERO.is_dir():
        check = subprocess.run([str(PY), "-c", "import torch, transformers, bitsandbytes, mujoco; assert torch.cuda.is_available(); assert mujoco.__version__ == '2.3.7'; assert torch.__version__.startswith('2.2.0')"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        warm = check.returncode == 0
        log.write(check.stdout); log.flush()
    if not warm:
        print("准备运行环境；本步骤不会重复 V3 分支实验。", flush=True)
        run([sys.executable, "-u", SOURCE / "scripts/colab_bootstrap.py", "--prepare-only", "--run-dir", ROOT / "setup"])
    else:
        print("检测到可复用的 V3 环境。", flush=True)
    # Install the pinned diagnostic code instead of reusing the old editable package.
    run([PY, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", SOURCE])
    run([PY, "-m", "pip", "install", "--no-deps", "peft==0.10.0"])
    env = dict(os.environ, HF_HOME="/content/cape-hf-cache", PYTHONUNBUFFERED="1",
               MUJOCO_GL="osmesa", PYOPENGL_PLATFORM="osmesa", TF_CPP_MIN_LOG_LEVEL="2")
    run([PY, "-u", SOURCE / "scripts/run_lora_update_v7.py",
         "--libero-root", LIBERO, "--output-root", ROOT / "rollouts", "--init-state-ids", "0"], env=env)
    state["status"] = "development_update_complete"
    print("完整回合开发验证完成；请上传 ZIP。这仍不是独立测试集上的适应结果。", flush=True)
except BaseException as exc:
    state["status"] = "failed"
    state["error"] = {"type": type(exc).__name__, "message": str(exc), "traceback": traceback.format_exc()}
    print(traceback.format_exc(), flush=True)
    raise
finally:
    (ROOT / "launcher_status.json").write_text(json.dumps(state, indent=2), encoding="utf-8")
    log.close()
    archive = shutil.make_archive(str(ROOT), "zip", ROOT)
    print("\n请把这个 ZIP 上传回对话：", archive, flush=True)
    from google.colab import files
    files.download(archive)
